In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import zipfile
from pathlib import Path
import pickle
import tarfile
import datetime
import numpy as np
import urllib.request
import sklearn.metrics
import tensorflow as tf
import torch
import keras
import matplotlib.pyplot as plt
import tensorboard as tb

import cv2

N_CLASSES = 200

In [2]:
# gpus = tf.config.experimental.list_physical_devices('GPU')
# if gpus:
#     print(gpus)
#     try:
#         for gpu in gpus:
#             print(gpu)
#             tf.config.experimental.set_memory_growth(gpu, True)
#     except RuntimeError as e:
#         print(e)

In [3]:
print(keras.backend.backend())
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"Current device index: {torch.cuda.current_device()}")

torch
CUDA available: True
GPU name: NVIDIA GeForce RTX 3060
Current device index: 0


In [4]:
BATCH_SIZE = 128
HISTORY_DIR = Path('./history')
HISTORY_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR_ROOT = Path('./dataset')
DATASET_DIR_ROOT.mkdir(parents=True, exist_ok=True)
DATASET_DIR = DATASET_DIR_ROOT / 'tiny-imagenet-200'

In [5]:
def download_data():
    dataset = DATASET_DIR
    dataset_zip = dataset.with_suffix('.zip')

    if not dataset_zip.exists():
        urllib.request.urlretrieve('http://cs231n.stanford.edu/tiny-imagenet-200.zip', dataset_zip)

    if not dataset.exists():
        file = zipfile.ZipFile(dataset_zip, 'r')
        file.extractall(path=DATASET_DIR_ROOT)

In [6]:
download_data()

In [7]:
with open(DATASET_DIR / "wnids.txt", "r") as fc:
    classes = [l.strip() for l in fc.readlines() ]
classes_map = { c: i for i, c in enumerate(classes) }
classes_map

{'n02124075': 0,
 'n04067472': 1,
 'n04540053': 2,
 'n04099969': 3,
 'n07749582': 4,
 'n01641577': 5,
 'n02802426': 6,
 'n09246464': 7,
 'n07920052': 8,
 'n03970156': 9,
 'n03891332': 10,
 'n02106662': 11,
 'n03201208': 12,
 'n02279972': 13,
 'n02132136': 14,
 'n04146614': 15,
 'n07873807': 16,
 'n02364673': 17,
 'n04507155': 18,
 'n03854065': 19,
 'n03838899': 20,
 'n03733131': 21,
 'n01443537': 22,
 'n07875152': 23,
 'n03544143': 24,
 'n09428293': 25,
 'n03085013': 26,
 'n02437312': 27,
 'n07614500': 28,
 'n03804744': 29,
 'n04265275': 30,
 'n02963159': 31,
 'n02486410': 32,
 'n01944390': 33,
 'n09256479': 34,
 'n02058221': 35,
 'n04275548': 36,
 'n02321529': 37,
 'n02769748': 38,
 'n02099712': 39,
 'n07695742': 40,
 'n02056570': 41,
 'n02281406': 42,
 'n01774750': 43,
 'n02509815': 44,
 'n03983396': 45,
 'n07753592': 46,
 'n04254777': 47,
 'n02233338': 48,
 'n04008634': 49,
 'n02823428': 50,
 'n02236044': 51,
 'n03393912': 52,
 'n07583066': 53,
 'n04074963': 54,
 'n01629819': 55,
 '

In [8]:
def get_dataset_part(path_to_dir_with_annotations: Path, is_val: bool = False) -> tuple[list[Path], list[int]]:
    _path = path_to_dir_with_annotations
    assert _path.is_dir()

    metadata_path = list(_path.glob("*.txt"))[0]

    with open(metadata_path, "r") as fm:
        _md = np.loadtxt(fm, dtype=str)
        file_paths = [_path / "images" / name for name in _md[:, 0]]
        if is_val:
            name_classes = list(map(str, _md[:, 1]))
        else:
            name_classes = [_path.name] * len(file_paths)

    i_classes = [classes_map[n] for n in name_classes]

    return file_paths, i_classes

def get_train_dataset(path_to_dir_with_train_dirs: Path) -> tuple[list[Path], list[int]]:
    _path = path_to_dir_with_train_dirs
    assert _path.is_dir()

    file_paths = []
    i_classes = []
    for class_dir in _path.iterdir():
        fps, cid = get_dataset_part(class_dir)
        file_paths.extend(fps)
        i_classes.extend(cid)

    return file_paths, i_classes

In [9]:
# get_dataset_part(DATASET_DIR / "val", is_val=True)
# get_dataset_part(DATASET_DIR / "train" / "n01443537")

# td = get_train_dataset(DATASET_DIR / "train")

In [10]:
# list(map(len, td))

In [11]:
class Dataset(keras.utils.PyDataset):

    def __init__(self, image_paths: list[Path], image_classes: list[int], batch_size: int, seed: int | None = None, shuffle: bool = False, **kwargs):
        super().__init__(**kwargs)
        self.shuffle = shuffle
        self.batch_size = batch_size
        self.seed = seed
        self.rng = np.random.default_rng(seed)

        self.image_paths = image_paths
        self.image_classes = np.array(image_classes)

        self.image_index = np.arange(len(image_paths))

        self.on_epoch_end()

    def make_train_dataset(batch_size: int, seed: int | None = None, shuffle: bool = False, **kwargs):
        return Dataset(*get_train_dataset(DATASET_DIR / "train"), batch_size, seed, shuffle, **kwargs)
    
    def make_validation_dataset(batch_size: int, seed: int | None = None, shuffle: bool = False, **kwargs):
        return Dataset(*get_dataset_part(DATASET_DIR / "val", is_val=True), batch_size, seed, shuffle, **kwargs)

    def __len__(self):
        return (len(self.image_paths) + self.batch_size - 1) // self.batch_size

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.image_index)
    
    def __getitem__(self, index: int):
        start = index * self.batch_size
        end = (index + 1) * self.batch_size
        batch = self.image_index[start:end]

        images = np.array([cv2.imread(self.image_paths[idx]) for idx in batch])
        return images , self.image_classes[batch]


In [12]:
train_dataset = Dataset.make_train_dataset(batch_size=BATCH_SIZE, seed=42, shuffle=True)

In [13]:
val_dataset = Dataset.make_validation_dataset(batch_size=BATCH_SIZE, seed=42, shuffle=False)

In [14]:
def res_block(inp, dim: int, name: str | None = None, is_first: bool=False):
    x = inp

    half_text = " :2" if is_first else ""

    x = keras.layers.Conv2D(dim // 4, 1, strides=1 + is_first, activation='relu', padding='same', name=f"{name}_conv_1{half_text}")(x)
    x = keras.layers.Conv2D(dim // 4, 3, padding='same', name=f"{name}_conv_2")(x)
    x = keras.layers.Conv2D(dim, 1, name=f"{name}_conv_3")(x)
    if (not is_first):
        x = keras.layers.Add(name=f"{name}_add")([x, inp])

    return x

def res_block2(inp, dim: int, name: str | None = None, is_first: bool=False):
    x = inp

    half_text = " :2" if is_first else ""

    x = keras.layers.Conv2D(dim // 4, 1, strides=1 + is_first, padding='same', name=f"{name}_conv_1{half_text}")(x)
    x = keras.layers.LayerNormalization(name=f"{name}_ln_1")(x)
    x = keras.layers.Activation('relu', name=f'{name}_relu_1')(x)
    x = keras.layers.Conv2D(dim // 4, 3, padding='same', name=f"{name}_conv_2")(x)
    x = keras.layers.LayerNormalization(name=f"{name}_ln_2")(x)
    x = keras.layers.Activation('relu', name=f"{name}_relu_2")(x)
    x = keras.layers.Conv2D(dim, 1, name=f"{name}_conv_3")(x)
    if (not is_first):
        x = keras.layers.Add(name=f"{name}_add")([x, inp])

    return x

In [15]:
x = inputs = keras.layers.Input((64, 64, 3), name='inp')

x = keras.layers.Conv2D(64, 7, strides=2, padding='same', name="inp_conv")(x)
x = keras.layers.MaxPool2D(3, strides=1, padding='same', name="inp_max")(x)

nums = [
    (64, 2),
    (128, 2),
    (256, 2),
    # (512, 2)
]

for i, (ker, times) in enumerate(nums):
    for j in range(times):
        x = res_block2(x, ker, name=f"conv_{i}_{j}", is_first=(j == 0 and i > 0))

x = keras.layers.AveragePooling2D(2, name="ad_avg_pool_end")(x)
x = keras.layers.Flatten(name="flatten")(x)
x = keras.layers.Dense(N_CLASSES, activation='softmax', name="dense_softmax")(x)

model = keras.models.Model(inputs, x)

In [16]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ inp (InputLayer)    │ (None, 64, 64, 3) │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_conv (Conv2D)   │ (None, 32, 32,    │      9,472 │ inp[0][0]         │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inp_max             │ (None, 32, 32,    │          0 │ inp_conv[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_0_0_conv_1     │ (None, 32, 32,    │      1,040 │ inp_max[0][0]     │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_0_0_ln_1       │ (None, 32, 32,    │         32 │ conv_0_0_conv_1[… │
│ (LayerNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_0_0_relu_1     │ (None, 32, 32,    │          0 │ conv_0_0_ln_1[0]… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_0_0_conv_2     │ (None, 32, 32,    │      2,320 │ conv_0_0_relu_1[… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_0_0_ln_2       │ (None, 32, 32,    │         32 │ conv_0_0_conv_2[… │
│ (LayerNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_0_0_relu_2     │ (None, 32, 32,    │          0 │ conv_0_0_ln_2[0]… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_0_0_conv_3     │ (None, 32, 32,    │      1,088 │ conv_0_0_relu_2[… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_0_0_add (Add)  │ (None, 32, 32,    │          0 │ conv_0_0_conv_3[… │
│                     │ 64)               │            │ inp_max[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_0_1_conv_1     │ (None, 32, 32,    │      1,040 │ conv_0_0_add[0][… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_0_1_ln_1       │ (None, 32, 32,    │         32 │ conv_0_1_conv_1[… │
│ (LayerNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_0_1_relu_1     │ (None, 32, 32,    │          0 │ conv_0_1_ln_1[0]… │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_0_1_conv_2     │ (None, 32, 32,    │      2,320 │ conv_0_1_relu_1[… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_0_1_ln_2       │ (None, 32, 32,    │         32 │ conv_0_1_conv_2[… │
│ (LayerNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_0_1_relu_2     │ (None, 32, 32,    │          0 │ conv_0_1_ln_2[0]… │
│ (Activation)        │ 16)               │            │                 

 Total params: 1,003,656 (3.83 MB)

 Trainable params: 1,003,656 (3.83 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
model.compile(
    optimizer=keras.optimizers.Nadam(),
    loss=keras.losses.SparseCategoricalCrossentropy(), 
    metrics=[
        keras.metrics.SparseCategoricalAccuracy()
    ]
)

In [18]:
# logdir = HISTORY_DIR / datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
logdir = HISTORY_DIR / "20260516-142626"
logdir.mkdir(parents=True, exist_ok=True)
logdir

WindowsPath('history/20260516-142626')

In [19]:
model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
    logdir / 'model.keras',
    save_best_only=True
)

In [27]:
tb_log = logdir / 'logs'
tb_log
tensorboard_callback = keras.callbacks.TensorBoard(
    tb_log,
)
tb_log_str = str(tb_log)

In [ ]:
%load_ext tensorboard

In [32]:
tb.notebook.list()

Known TensorBoard instances:
  - port 6007: logdir history20260516-142626logs (started 0:37:48 ago; pid 11196)
  - port 6006: logdir history20260516-142626 (started 1:15:36 ago; pid 12624)
  - port 6007: logdir history20260514-232528 (started 1 day, 16:16:53 ago; pid 15996)
  - port 6006: logdir history20260514-215558 (started 1 day, 17:46:22 ago; pid 19032)
  - port 6006: logdir history20260514-210843 (started 1 day, 18:33:06 ago; pid 20772)
  - port 6008: logdir history20260515-000655 (started 1 day, 15:35:23 ago; pid 21672)
  - port 6009: logdir history20260515-002841 (started 1 day, 15:13:23 ago; pid 22020)
  - port 6006: logdir history20260514-225019 (started 1 day, 16:52:01 ago; pid 3256)
  - port 6006: logdir history20260514-222705 (started 1 day, 17:15:11 ago; pid 8736)
  - port 6006: logdir history/20260516-142626/logs (started 0:16:33 ago; pid 9692)


In [33]:
%tensorboard --logdir history/20260516-142626/logs

Reusing TensorBoard on port 6006 (pid 9692), started 0:16:36 ago. (Use '!kill 9692' to kill it.)

In [80]:
# import gc
# gc.collect()
# torch.cuda.empty_cache()

In [ ]:
# model.load_weights(r"history\20260516-142626\model.keras")

d:\Program Files\GitHub\CompMath2\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'nadam', because it has 2 variables whereas the saved optimizer has 131 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
model.fit(
    train_dataset, 
    validation_data=val_dataset, 
    batch_size=BATCH_SIZE, 
    epochs=20,
    callbacks=[
        model_checkpoint_callback, 
        tensorboard_callback
    ]
)

Epoch 1/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 73s 92ms/step - loss: 2.5387 - sparse_categorical_accuracy: 0.4004 - val_loss: 3.2236 - val_sparse_categorical_accuracy: 0.2872
Epoch 2/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 65s 83ms/step - loss: 2.3801 - sparse_categorical_accuracy: 0.4295 - val_loss: 3.3176 - val_sparse_categorical_accuracy: 0.2786
Epoch 3/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 65s 84ms/step - loss: 2.2338 - sparse_categorical_accuracy: 0.4566 - val_loss: 3.3843 - val_sparse_categorical_accuracy: 0.2775
Epoch 4/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 66s 84ms/step - loss: 2.0830 - sparse_categorical_accuracy: 0.4878 - val_loss: 3.4365 - val_sparse_categorical_accuracy: 0.2819
Epoch 5/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 65s 83ms/step - loss: 1.9443 - sparse_categorical_accuracy: 0.5161 - val_loss: 3.4936 - val_sparse_categorical_accuracy: 0.2843
Epoch 6/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 65s 83ms/step - loss: 1.8077 - sparse_categorical_accuracy: 0.5441 - val_loss: 3.6553 - val_sparse_categorical_accuracy:

In [29]:
y_true = val_dataset.image_classes
y_pred = np.argmax(model.predict(val_dataset), axis=1)

79/79 ━━━━━━━━━━━━━━━━━━━━ 4s 53ms/step


In [88]:
_, ax = plt.subplots(figsize=(75, 75))
sklearn.metrics.ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=ax, colorbar=False)

plt.tight_layout()
plt.savefig(logdir / 'valid.png')

In [30]:
sklearn.metrics.accuracy_score(y_true, y_pred)

0.2489